<a href="https://colab.research.google.com/github/Fawwaz-05/data-science-toolkit2/blob/main/Tutorial6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd


file = '/content/drive/MyDrive/Dataset/salary.csv', '/content/drive/MyDrive/Dataset/Tutorial 6 Data Processing 2-20261118 (Unzipped Files)/xtra_info dataset.csv'
df_salary = pd.read_csv(file[0])
df_extra = pd.read_csv(file[1])

In [3]:
from sklearn.preprocessing import MinMaxScaler , StandardScaler


#print shape
print("Salary table size" , df_salary.shape)
print("Extra table size " , df_extra.shape)

#display a few row
display(df_salary.head(3))
display(df_extra.head(3))

Salary table size (28180, 19)
Extra table size  (13404, 4)


,Unnamed: 0,Timestamp,How old are you?,What industry do you work in?,Job title,"If your job title needs additional context, please clarify here:","What is your annual salary? (You'll indicate the currency in a later question. If you are part-time or hourly, please enter an annualized equivalent -- what you would earn if you worked the job 40 hours a week, 52 weeks a year.)","How much additional monetary compensation do you get, if any (for example, bonuses or overtime in an average year)? Please only include monetary compensation here, not the value of benefits.",Please indicate the currency,"If ""Other,"" please indicate the currency here:","If your income needs additional context, please provide it here:",What country do you work in?,"If you're in the U.S., what state do you work in?",What city do you work in?,How many years of professional work experience do you have overall?,How many years of professional work experience do you have in your field?,What is your highest level of education completed?,What is your gender?,What is your race? (Choose all that apply.)
0,0,2021-04-27 11:02:09.743,25-34,Education (Higher Education),Research and Instruction Librarian,NaN,55000,0.0,USD,NaN,NaN,United States,Massachusetts,Boston,5-7 years,5-7 years,Master's degree,Woman,White
1,1,2021-04-27 11:02:21.562,25-34,Computing or Tech,Change & Internal Communications Manager,NaN,54600,4000.0,GBP,NaN,NaN,United Kingdom,NaN,Cambridge,8 - 10 years,5-7 years,College degree,Non-binary,White
2,2,2021-04-27 11:02:38.125,25-34,"Accounting, Banking & Finance",Marketing Specialist,NaN,34000,NaN,USD,NaN,NaN,US,Tennessee,Chattanooga,2 - 4 years,2 - 4 years,College degree,Woman,White


,USERID,CGPA,Salary_month_USD,Special Bonus_GBP
0,8683,2.990627,4583.333333,2437.0
1,3880,2.288131,5166.666667,740.0
2,8898,3.903324,5000.000000,3806.0


In [4]:

#rename Columns

short_name_salary = {

              df_salary.columns[0]: 'USERID',
              df_salary.columns[6]: 'Annual_Salary',
              df_salary.columns[8]: 'Currency',
              df_salary.columns[7]: 'Extra_Compensation',
              df_salary.columns[11]: 'Country' ,
              df_salary.columns[13]: 'City' ,
              }



#apply
df_salary.rename(columns=short_name_salary , inplace=True)


#apply and make sure all data in specific data type
df_salary["USERID"] = pd.to_numeric(df_salary["USERID"] , errors='coerce')

#delete "," is any
df_salary["Annual_Salary"] = pd.to_numeric(df_salary["Annual_Salary"].astype(str).str.replace(",","",regex=False) , errors='coerce')

df_extra["USERID"] = pd.to_numeric(df_extra["USERID"] , errors='coerce')


display(df_salary[["USERID" ,"Annual_Salary", "Currency" , "Country" , "City"]])




,USERID,Annual_Salary,Currency,Country,City
0,0,55000,USD,United States,Boston
1,1,54600,GBP,United Kingdom,Cambridge
2,2,34000,USD,US,Chattanooga
3,3,62000,USD,USA,Milwaukee
4,4,60000,USD,US,Greenville
...,...,...,...,...,...
28175,28175,200000,USD,USA,NYC
28176,28176,0,USD,USA,lansing
28177,28177,1,EUR,1,1
28178,28178,41600,USD,U.S.,Tucson


In [5]:

#count the total of row and how many the unique value
print("Extra rows",len(df_extra))
print("Unique extra id" , df_extra["USERID"].nunique())

#kira sum duplicate
print("Repeated id " , df_extra.duplicated(subset="USERID").sum())

# keep first row of number
df_extra_unique = df_extra.drop_duplicates(subset="USERID" , keep = "first").copy()


#check user id
print("After remove " , len(df_extra_unique))

assert not df_extra_unique.duplicated(subset="USERID").any()



Extra rows 13404
Unique extra id 8442
Repeated id  4962
After remove  8442


In [6]:


# merge dataframe

df_integrated = pd.merge (
    df_salary ,
    df_extra_unique ,
    on="USERID" ,
    how="left" , #keep every row on original
    validate="one_to_one" # check every id and make sure it unique
)

print("Salary row before" , len(df_salary))
print("Salary row after" , len(df_integrated))


print("Rows with extra information" , df_integrated["CGPA"].notna().sum())


assert len(df_salary) == len(df_integrated)

display(df_integrated[["USERID" , "Annual_Salary" , "Currency" , "CGPA" , "Salary_month_USD" , "Special Bonus_GBP"]].head(10))


Salary row before 28180
Salary row after 28180
Rows with extra information 8442


,USERID,Annual_Salary,Currency,CGPA,Salary_month_USD,Special Bonus_GBP
0,0,55000,USD,NaN,NaN,NaN
1,1,54600,GBP,2.695615,4250.000000,125.0
2,2,34000,USD,NaN,NaN,NaN
3,3,62000,USD,NaN,NaN,NaN
4,4,60000,USD,2.689383,12500.000000,1133.0
5,5,62000,USD,NaN,NaN,NaN
6,6,33000,USD,3.541300,6333.333333,878.0
7,7,50000,USD,NaN,NaN,NaN
8,8,112000,USD,NaN,NaN,NaN
9,9,45000,USD,2.819416,12500.000000,4829.0


In [7]:


comparison = df_integrated.loc[
    (df_integrated["Currency"] == "USD") &
    df_integrated["Salary_month_USD"].notna(),
    ["USERID", "Annual_Salary", "Salary_month_USD"]
].copy()


comparison["Calculated_Montly_USD"] = comparison["Annual_Salary"]/ 12

rough_match = (

               comparison["Salary_month_USD"] - comparison["Calculated_Montly_USD"] ).abs() < 0.01
print("Number of comparable record : " , len(comparison))
print("Mathching montly amounts : ", rough_match.sum())
print("Nonmatching montly amounts " , (~rough_match).sum()) # reverse the result

display(comparison.head())

Number of comparable record :  7063
Mathching montly amounts :  55
Nonmatching montly amounts  7008


,USERID,Annual_Salary,Salary_month_USD,Calculated_Montly_USD
4,4,60000,12500.000000,5000.000000
6,6,33000,6333.333333,2750.000000
9,9,45000,12500.000000,3750.000000
10,10,47500,4583.333333,3958.333333
12,12,100000,12083.333330,8333.333333


In [8]:
df_integrated["Annual_Salary_Div_12"] = df_integrated["Annual_Salary"] / 12

df_integrated = df_integrated.drop(columns=["Annual_Salary_Div_12"])


print("Derived example column removed : " ,
      "Annual_Salary_Div_12" not in df_integrated.columns)

Derived example column removed :  True


In [9]:
GBP = 1.30

df_integrated["Special Bonus_USD"] = (
    pd.to_numeric(df_integrated["Special Bonus_GBP"], errors="coerce") * GBP
)

# 'Salary_month_GBP' does not exist in df_integrated, so we skip dropping it
# Alternatively, use: df_integrated.drop(columns=["Salary_month_GBP"], errors="ignore")

# Corrected column name spelling: "Special Bonus_USD"
display(df_integrated.loc[df_integrated["Special Bonus_USD"].notna(), ["USERID", "Special Bonus_USD"]].head())


,USERID,Special Bonus_USD
1,1,162.5
4,4,1472.9
6,6,1141.4
9,9,6277.7
10,10,1016.6


In [10]:
#Normalization


df_usd = df_integrated[df_integrated["Currency"] == "USD"].copy()


df_usd =df_usd.loc[df_usd["Annual_Salary"].notna() & (df_usd["Annual_Salary"] > 0)].copy()


print("USD Salary Record " , len(df_usd))
display(df_usd[["Annual_Salary"]].head())





USD Salary Record  23441


,Annual_Salary
0,55000
2,34000
3,62000
4,60000
5,62000


In [12]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Initialize scale tools
minmax_scaler = MinMaxScaler()
zscore_scaler = StandardScaler()

# Apply MinMax Scaler
df_usd["Salary_MinMax"] = minmax_scaler.fit_transform(df_usd[["Annual_Salary"]]).ravel()

# Fit the scaler using the annual USD salaries and transform them.
df_usd["Salary_Zscore"] = zscore_scaler.fit_transform(
    df_usd[["Annual_Salary"]]
).ravel()

# Compare the original salary with both scaling results.
display(df_usd[["Annual_Salary", "Salary_MinMax", "Salary_Zscore"]].head(10))

# Calculate the average of the standardized salaries
print("Mean:", df_usd["Salary_Zscore"].mean())

# Calculate how spread out the standardized salaries are
print("Standard deviation:", df_usd["Salary_Zscore"].std())

,Annual_Salary,Salary_MinMax,Salary_Zscore
0,55000,0.000539,-0.058724
2,34000,0.000333,-0.086545
3,62000,0.000608,-0.049450
4,60000,0.000588,-0.052100
5,62000,0.000608,-0.049450
6,33000,0.000324,-0.087870
7,50000,0.000490,-0.065348
8,112000,0.001098,0.016792
9,45000,0.000441,-0.071972
10,47500,0.000466,-0.068660


Mean: -9.699828311216759e-18
Standard deviation: 1.0000213308305577


In [13]:
# Remove rows with missing/empty city names.
df_city = df_usd.dropna(subset=["City"])

# This code groups employees by their country and city, then calculates salary statistics for each location.
city_summary = df_city.groupby(["Country", "City"], as_index=False).agg(
    Number_of_Records=("Annual_Salary", "count"),  # How many reported salaries?
    Mean_Salary_USD=("Annual_Salary", "mean"),     # Average annual salary
    Median_Salary_USD=("Annual_Salary", "median"), # Middle annual salary
    Minimum_Salary_USD=("Annual_Salary", "min"),   # Smallest annual salary
    Maximum_Salary_USD=("Annual_Salary", "max")    # Largest annual salary
)

# Display a sample of the grouped results.
display(city_summary.head(10))

,Country,City,Number_of_Records,Mean_Salary_USD,Median_Salary_USD,Minimum_Salary_USD,Maximum_Salary_USD
0,U.S.,indianapolis,1,42500.0,42500.0,42500,42500
1,US,Philadelphia,1,86000.0,86000.0,86000,86000
2,United States,South Bend,1,69000.0,69000.0,69000,69000
3,Africa,Africa,1,90000.0,90000.0,90000,90000
4,America,Ann Arbor,1,34000.0,34000.0,34000,34000
5,America,Atlanta,1,45000.0,45000.0,45000,45000
6,America,Boston,1,194000.0,194000.0,194000,194000
7,America,Chicago,1,70.0,70.0,70,70
8,America,Columbia,1,75000.0,75000.0,75000,75000
9,America,Houston,1,20488.0,20488.0,20488,20488


In [14]:
# Sort all groups by average annual salary, highest to lowest.
all_cities_sorted = city_summary.sort_values(
    by="Mean_Salary_USD", ascending=False
)

# Show the ten cities with the highest sample mean salary.
print("Top 10 city/country groups (all sizes):")
display(all_cities_sorted.head(10))

Top 10 city/country groups (all sizes):


,Country,City,Number_of_Records,Mean_Salary_USD,Median_Salary_USD,Minimum_Salary_USD,Maximum_Salary_USD
62,Colombia,Bogota,2,51012500.0,51012500.0,25000,102000000
64,Colombia,Cali,2,26106000.0,26106000.0,12000,52200000
6924,usa,Rice,1,10000000.0,10000000.0,10000000,10000000
6609,Usa,Pinebrook,1,5000044.0,5000044.0,5000044,5000044
59,China,Hong Kong,1,5000000.0,5000000.0,5000000,5000000
2524,USA,Newyork,1,2600000.0,2600000.0,2600000,2600000
175,Taiwan,Milwaukee,1,2000000.0,2000000.0,2000000,2000000
102,India,Banglore,1,1000000.0,1000000.0,1000000,1000000
3639,United Stated,Spokane,1,832000.0,832000.0,832000,832000
430,U.S.,Salina,1,734400.0,734400.0,734400,734400
